<a href="https://colab.research.google.com/github/JennyDuda/Mall-Customers-Segmenta-o-de-Clientes/blob/main/Segmenta%C3%A7%C3%A3oClientes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛍 Mall Customers — Segmentação de Clientes

## Objetivo de negócio
Um shopping quer entender **quem são seus clientes** e **como agir diferente com cada grupo**.  
A partir de dados demográficos e comportamentais de 200 clientes, vamos descobrir segmentos naturais e traduzi-los em recomendações acionáveis de marketing.

## Estrutura do notebook
1. Setup e carregamento dos dados
2. Limpeza e feature engineering
3. Análise exploratória (EDA) — *Quem são nossos clientes?*
4. Definição do número ideal de clusters
5. Clustering com K-Means — *Existem grupos naturais?*
6. Perfil e interpretação dos clusters
7. Validação da estabilidade dos clusters (substituindo a classificação supervisionada)
8. Visualização avançada (PCA, t-SNE aplicados corretamente)
9. Insights e recomendações de negócio

## Decisões técnicas principais
| Escolha | Justificativa |
|---|---|
| K-Means com k=5 | Validado por Elbow + Silhouette; clusters bem separados no espaço renda × score |
| StandardScaler antes do clustering | K-Means usa distância euclidiana — sem padronização, renda (~15–137) domina sobre score (1–99) |
| `Gender` como `map()`, não `LabelEncoder` | Variável binária: `map` é explícito e evita ordenação implícita acidental |
| Validação por estabilidade (bootstrap) | Mede se os clusters são robustos, não apenas se um modelo consegue imitá-los |
| PCA/t-SNE em 4 features | Redução dimensional só faz sentido quando há mais dimensões do que o eixo de visualização |

## Limitações
- Dataset pequeno (200 registros): resultados são ilustrativos, não generalizáveis sem mais dados
- Apenas 3 variáveis de comportamento: adicionar histórico de compras e frequência tornaria os clusters mais ricos
- K-Means assume clusters esféricos: DBSCAN ou GMM podem revelar formas diferentes — com 200 registros, o ganho seria marginal


## 1. Setup e carregamento dos dados

In [ ]:
# =============================================================
# IMPORTS E CONFIGURAÇÃO DE ESTILO
# =============================================================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.spatial.distance import cdist

warnings.filterwarnings('ignore')

# Estilo profissional consistente
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 15,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Paleta acessível (segura para daltonismo)
PALETTE = sns.color_palette('colorblind', 10)
CLUSTER_COLORS = {i: PALETTE[i] for i in range(10)}

print("✅ Ambiente configurado.")
print(f"   pandas {pd.__version__} | numpy {np.__version__} | seaborn {sns.__version__}")


In [ ]:
# =============================================================
# CARREGAMENTO DO DATASET — com fallback local
# =============================================================
import os

def load_dataset():
    """Tenta kagglehub; se falhar, procura CSV local."""
    # Tentativa 1: kagglehub (Colab / ambiente com credenciais Kaggle)
    try:
        import kagglehub
        path = kagglehub.dataset_download("abdallahwagih/mall-customers-segmentation")
        csv_path = os.path.join(path, "Mall_Customers.csv")
        print(f"✅ Dataset baixado via KaggleHub: {csv_path}")
        return pd.read_csv(csv_path)
    except Exception as e:
        print(f"⚠️  KaggleHub falhou ({e})")

    # Tentativa 2: arquivo local (se o usuário baixou manualmente)
    for local_path in ["Mall_Customers.csv", "data/Mall_Customers.csv", "Data/Mall_Customers.csv"]:
        if os.path.exists(local_path):
            print(f"✅ Dataset carregado localmente: {local_path}")
            return pd.read_csv(local_path)

    raise FileNotFoundError(
        "Dataset não encontrado. Faça o download em:\n"
        "https://www.kaggle.com/datasets/abdallahwagih/mall-customers-segmentation\n"
        "e salve como 'Mall_Customers.csv' na mesma pasta deste notebook."
    )

df_raw = load_dataset()
print(f"\n📊 Shape: {df_raw.shape}")
df_raw.head()


## 2. Limpeza e feature engineering

Antes de qualquer análise, entendemos a estrutura, verificamos qualidade e criamos as features necessárias.

In [ ]:
# =============================================================
# RENOMEAÇÃO E INSPEÇÃO INICIAL
# =============================================================
df = df_raw.copy()

df.rename(columns={
    'CustomerID':           'CustomerID',
    'Genre':                'Gender',         # nome original no dataset
    'Age':                  'Age',
    'Annual Income (k$)':   'Annual_Income',
    'Spending Score (1-100)': 'Spending_Score'
}, inplace=True)

print("=== Tipos e primeiras linhas ===")
display(df.dtypes.to_frame('dtype').T)
display(df.head())

print("\n=== Qualidade dos dados ===")
quality = pd.DataFrame({
    'missing': df.isnull().sum(),
    'missing_%': (df.isnull().sum() / len(df) * 100).round(1),
    'duplicados': [df.duplicated().sum()] + ['-'] * (len(df.columns) - 1),
    'únicos': df.nunique(),
})
display(quality)
print("\n✅ Nenhum valor ausente ou duplicado — dataset limpo.")


In [ ]:
# =============================================================
# FEATURE ENGINEERING
# =============================================================

# Gênero: map explícito — evita ordenação implícita do LabelEncoder
# (Female=0, Male=1 por convenção binária; documentado aqui)
df['Gender_Code'] = df['Gender'].map({'Female': 0, 'Male': 1})

# Faixas etárias interpretáveis para negócio
bins   = [0, 20, 30, 40, 50, 60, 100]
labels = ['< 20', '20–29', '30–39', '40–49', '50–59', '60+']
df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels)

# Padronização (necessária para K-Means — usa distância euclidiana)
scaler = StandardScaler()
df[['Age_sc', 'Income_sc', 'Score_sc']] = scaler.fit_transform(
    df[['Age', 'Annual_Income', 'Spending_Score']]
)

print("✅ Features criadas:")
print(f"   Gender_Code: {df['Gender_Code'].value_counts().to_dict()}")
print(f"   Age_Group  : {df['Age_Group'].value_counts().sort_index().to_dict()}")
print(f"   Variáveis padronizadas: Age_sc, Income_sc, Score_sc (μ≈0, σ≈1)")
display(df[['Age_sc', 'Income_sc', 'Score_sc']].describe().round(2))


## 3. Análise exploratória — *Quem são nossos clientes?*

Buscamos padrões nos dados brutos antes de qualquer modelagem. Cada gráfico responde a uma pergunta de negócio.

In [ ]:
# =============================================================
# EDA — VISÃO GERAL EM PAINEL ÚNICO
# Pergunta: Qual é o perfil demográfico base da nossa base?
# =============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Perfil demográfico e comportamental dos clientes", y=1.01)

# 1. Gênero
ax = axes[0, 0]
gender_counts = df['Gender'].value_counts()
bars = ax.bar(gender_counts.index, gender_counts.values,
              color=[PALETTE[0], PALETTE[1]], width=0.5, edgecolor='white')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height()}
({bar.get_height()/len(df)*100:.0f}%)',
            ha='center', va='bottom', fontsize=10)
ax.set_title("Distribuição de gênero")
ax.set_ylabel("Clientes")
ax.set_ylim(0, gender_counts.max() * 1.2)

# 2. Faixa etária
ax = axes[0, 1]
age_counts = df['Age_Group'].value_counts().sort_index()
ax.bar(age_counts.index.astype(str), age_counts.values,
       color=PALETTE[2], edgecolor='white')
ax.set_title("Distribuição por faixa etária")
ax.set_ylabel("Clientes")
ax.tick_params(axis='x', rotation=0)

# 3. Distribuição de Renda
ax = axes[0, 2]
ax.hist(df['Annual_Income'], bins=15, color=PALETTE[3], edgecolor='white', alpha=0.85)
ax.axvline(df['Annual_Income'].mean(),   color='red',   ls='--', lw=1.5,
           label=f"Média: ${df['Annual_Income'].mean():.0f}k")
ax.axvline(df['Annual_Income'].median(), color='navy', ls='--', lw=1.5,
           label=f"Mediana: ${df['Annual_Income'].median():.0f}k")
ax.set_title("Distribuição de renda anual")
ax.set_xlabel("Renda (k$)")
ax.set_ylabel("Clientes")
ax.legend(fontsize=9)

# 4. Distribuição de Spending Score
ax = axes[1, 0]
ax.hist(df['Spending_Score'], bins=15, color=PALETTE[4], edgecolor='white', alpha=0.85)
ax.axvline(df['Spending_Score'].mean(),   color='red',   ls='--', lw=1.5,
           label=f"Média: {df['Spending_Score'].mean():.0f}")
ax.axvline(df['Spending_Score'].median(), color='navy', ls='--', lw=1.5,
           label=f"Mediana: {df['Spending_Score'].median():.0f}")
ax.set_title("Distribuição de spending score")
ax.set_xlabel("Score (1–100)")
ax.set_ylabel("Clientes")
ax.legend(fontsize=9)

# 5. Renda × Score (scatter com gênero)
ax = axes[1, 1]
for gender, color in [('Female', PALETTE[0]), ('Male', PALETTE[1])]:
    sub = df[df['Gender'] == gender]
    ax.scatter(sub['Annual_Income'], sub['Spending_Score'],
               color=color, alpha=0.7, s=60, label=gender, edgecolors='white', lw=0.5)
ax.set_title("⚡ Renda vs. Score — padrão em 'X' sugere clusters")
ax.set_xlabel("Renda anual (k$)")
ax.set_ylabel("Spending score")
ax.legend()

# 6. Correlação
ax = axes[1, 2]
corr = df[['Age', 'Annual_Income', 'Spending_Score']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            annot_kws={'size': 11})
ax.set_title("Mapa de correlação")

plt.tight_layout()
plt.savefig("eda_overview.png", dpi=120, bbox_inches='tight')
plt.show()
print("💡 Destaque: o scatter Renda × Score já antecipa os 5 grupos naturais.")


In [ ]:
# =============================================================
# EDA — RENDA E SCORE POR FAIXA ETÁRIA
# Pergunta: Grupos etários têm perfis de gasto distintos?
# =============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Renda e comportamento de gasto por faixa etária")

for ax, col, label in zip(
    axes,
    ['Annual_Income', 'Spending_Score'],
    ['Renda anual (k$)', 'Spending Score']
):
    age_stats = df.groupby('Age_Group', observed=True)[col].agg(['mean','std']).reset_index()
    bars = ax.bar(age_stats['Age_Group'].astype(str), age_stats['mean'],
                  yerr=age_stats['std'], capsize=4,
                  color=PALETTE[5], edgecolor='white', error_kw={'lw':1.2})
    ax.set_title(f"{label} por faixa etária (média ± dp)")
    ax.set_xlabel("Faixa etária")
    ax.set_ylabel(label)
    ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()
print("💡 Spending score cai com a idade — jovens gastam mais proporcionalmente.")


## 4. Definição do número ideal de clusters

Usamos dois critérios complementares:
- **Elbow (inércia)**: busca o ponto de "cotovelo" onde adicionar mais clusters retorna pouco ganho
- **Silhouette Score**: mede a coesão interna vs. separação entre clusters (quanto maior, melhor)

In [ ]:
# =============================================================
# ELBOW + SILHOUETTE — escolha de k
# =============================================================
X_cluster = df[['Income_sc', 'Score_sc']]  # features usadas no clustering

k_range = range(2, 11)
inertia, sil_scores = [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cluster)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(X_cluster, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Escolha do número de clusters")

# Elbow
ax = axes[0]
ax.plot(list(k_range), inertia, 'o-', color=PALETTE[0], lw=2, ms=7)
ax.axvline(5, color='red', ls='--', lw=1.5, alpha=0.7, label='k=5 selecionado')
ax.set_title("Método Elbow (inércia)")
ax.set_xlabel("Número de clusters (k)")
ax.set_ylabel("Inércia (WCSS)")
ax.legend()

# Silhouette
ax = axes[1]
ax.plot(list(k_range), sil_scores, 's-', color=PALETTE[1], lw=2, ms=7)
best_k = k_range[np.argmax(sil_scores)]
ax.axvline(best_k, color='green', ls='--', lw=1.5, alpha=0.7, label=f'Melhor silhouette: k={best_k}')
ax.axvline(5, color='red',   ls='--', lw=1.5, alpha=0.7, label='k=5 selecionado')
ax.set_title("Silhouette Score")
ax.set_xlabel("Número de clusters (k)")
ax.set_ylabel("Silhouette Score")
ax.legend()

plt.tight_layout()
plt.show()

print(f"Melhor Silhouette Score individual: k={best_k} ({max(sil_scores):.3f})")
print(f"Silhouette para k=5: {sil_scores[3]:.3f}")
print()
print("📌 Decisão: k=5")
print("   • O Elbow mostra cotovelo claro em k=5")
print("   • k=5 gera 5 segmentos interpretáveis e acionáveis de negócio")
print(f"  • Silhouette de {sil_scores[3]:.3f} indica boa separação (> 0.5 = bom)")


## 5. Clustering com K-Means

Com k=5 definido, rodamos o modelo final e visualizamos os clusters.

In [ ]:
# =============================================================
# K-MEANS FINAL
# =============================================================
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_cluster)

# Rótulos de negócio (definidos após análise do perfil — seção 6)
CLUSTER_LABELS = {
    0: 'Renda média, gasto médio',
    1: 'Renda alta, gasto baixo',
    2: 'Renda baixa, gasto baixo',
    3: 'Renda alta, gasto alto',
    4: 'Renda baixa, gasto alto',
}
df['Cluster_Label'] = df['Cluster'].map(CLUSTER_LABELS)

sil_global = silhouette_score(X_cluster, df['Cluster'])
print(f"✅ K-Means ajustado | Silhouette Score global: {sil_global:.3f}")
print()
print(df['Cluster_Label'].value_counts().to_string())


In [ ]:
# =============================================================
# VISUALIZAÇÃO DOS CLUSTERS (espaço original Renda × Score)
# =============================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Clusters de clientes: Renda × Spending Score")

# Scatter colorido por cluster
ax = axes[0]
for c in sorted(df['Cluster'].unique()):
    sub = df[df['Cluster'] == c]
    ax.scatter(sub['Annual_Income'], sub['Spending_Score'],
               color=PALETTE[c], label=f'C{c}: {CLUSTER_LABELS[c]}',
               s=80, alpha=0.85, edgecolors='white', lw=0.5)

# Centroides no espaço original (desnormalizados)
centroids_sc = kmeans.cluster_centers_
centroids_orig = scaler.inverse_transform(
    np.column_stack([
        np.zeros(5),            # Age_sc placeholder
        centroids_sc[:, 0],     # Income_sc
        centroids_sc[:, 1],     # Score_sc
    ])
)[:, 1:]  # só Income e Score

ax.scatter(centroids_orig[:, 0], centroids_orig[:, 1],
           c='black', marker='X', s=200, zorder=5, label='Centroides')
ax.set_title("Clusters com centroides")
ax.set_xlabel("Renda anual (k$)")
ax.set_ylabel("Spending Score")
ax.legend(fontsize=8, loc='upper left')

# Boxplot de Score por cluster
ax = axes[1]
cluster_order = sorted(df['Cluster'].unique())
data_by_cluster = [df[df['Cluster'] == c]['Spending_Score'].values for c in cluster_order]
bp = ax.boxplot(data_by_cluster, patch_artist=True, notch=False,
                medianprops=dict(color='black', lw=2))
for patch, c in zip(bp['boxes'], cluster_order):
    patch.set_facecolor(PALETTE[c])
    patch.set_alpha(0.8)
ax.set_xticklabels([f'C{c}' for c in cluster_order])
ax.set_title("Distribuição de Spending Score por cluster")
ax.set_xlabel("Cluster")
ax.set_ylabel("Spending Score")

plt.tight_layout()
plt.savefig("clusters_scatter.png", dpi=120, bbox_inches='tight')
plt.show()


## 6. Perfil e interpretação dos clusters

Aqui entendemos *quem* é cada grupo e *o que isso significa* para o negócio.

In [ ]:
# =============================================================
# SUMÁRIO ESTATÍSTICO POR CLUSTER
# =============================================================
cluster_summary = df.groupby('Cluster').agg(
    N=('Cluster', 'count'),
    Idade_media=('Age', 'mean'),
    Renda_media=('Annual_Income', 'mean'),
    Score_medio=('Spending_Score', 'mean'),
    Pct_Feminino=('Gender_Code', lambda x: (x == 0).mean() * 100),
).round(1)
cluster_summary['Rotulo'] = cluster_summary.index.map(CLUSTER_LABELS)
cluster_summary = cluster_summary[['Rotulo', 'N', 'Idade_media', 'Renda_media', 'Score_medio', 'Pct_Feminino']]
display(cluster_summary.rename(columns={
    'Rotulo': 'Perfil',
    'Idade_media': 'Idade (avg)',
    'Renda_media': 'Renda avg ($k)',
    'Score_medio': 'Score avg',
    'Pct_Feminino': 'Feminino (%)',
}))


In [ ]:
# =============================================================
# SILHOUETTE PLOT — detalhe por amostra
# Mostra a coesão de cada ponto dentro do seu cluster
# =============================================================
sil_vals = silhouette_samples(X_cluster, df['Cluster'])

fig, ax = plt.subplots(figsize=(9, 6))
y_lower = 10

for c in sorted(df['Cluster'].unique()):
    c_sil = np.sort(sil_vals[df['Cluster'] == c])
    size_c = len(c_sil)
    y_upper = y_lower + size_c
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil,
                     alpha=0.75, color=PALETTE[c], label=f'C{c}')
    ax.text(-0.05, y_lower + size_c / 2, f'C{c}', ha='right', va='center', fontsize=9)
    y_lower = y_upper + 10

ax.axvline(sil_global, color='red', ls='--', lw=1.5,
           label=f'Silhouette global: {sil_global:.3f}')
ax.set_title("Silhouette Plot — coesão por amostra e por cluster")
ax.set_xlabel("Silhouette coefficient")
ax.set_ylabel("Amostras agrupadas por cluster")
ax.set_yticks([])
ax.legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig("silhouette_plot.png", dpi=120, bbox_inches='tight')
plt.show()
print("💡 Barras longas = pontos bem alocados. Barras negativas = pontos borderline.")


In [ ]:
# =============================================================
# RADAR CHART — perfil comparativo dos clusters
# =============================================================
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

# Normaliza as métricas para [0,1] para o radar
metrics = ['Idade_media', 'Renda_media', 'Score_medio']
radar_labels = ['Idade', 'Renda', 'Score']
df_radar = cluster_summary[metrics].copy()
df_radar_norm = (df_radar - df_radar.min()) / (df_radar.max() - df_radar.min())

N = len(metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # fechar o polígono

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.set_title("Radar de perfil dos clusters (valores normalizados)", pad=20)

for c in sorted(df['Cluster'].unique()):
    vals = df_radar_norm.loc[c].tolist() + df_radar_norm.loc[c].tolist()[:1]
    ax.plot(angles, vals, 'o-', lw=2, color=PALETTE[c], label=f'C{c}')
    ax.fill(angles, vals, alpha=0.07, color=PALETTE[c])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=11)
ax.set_yticklabels([])
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

plt.tight_layout()
plt.show()


## 7. Validação da estabilidade dos clusters (bootstrap)

> **Por que não usar classificação supervisionada aqui?**  
> Treinar um classificador para prever os clusters do K-Means e obter 96% de acurácia é uma **validação circular**: o modelo aprende a imitar o K-Means, não a identificar padrões de negócio reais. É o equivalente de usar a gabarito como treino e teste ao mesmo tempo.  
>  
> A pergunta correta é: **os clusters são estáveis?** Se rodarmos o K-Means em subamostras diferentes dos dados, os mesmos grupos aparecem?  
> Medimos isso com **bootstrap**: 100 re-amostras com reposição → Adjusted Rand Index (ARI) entre cada resultado e o clustering original. ARI ≈ 1 = clusters idênticos.

In [ ]:
# =============================================================
# BOOTSTRAP STABILITY — ARI médio
# Referência: Hennig (2007) — Cluster-wise assessment of cluster stability
# =============================================================
from sklearn.metrics import adjusted_rand_score

N_BOOTSTRAP = 100
ari_scores = []

rng = np.random.default_rng(42)
X_np = X_cluster.values
labels_original = df['Cluster'].values

for _ in range(N_BOOTSTRAP):
    # Reamostragem com reposição
    idx = rng.choice(len(X_np), size=len(X_np), replace=True)
    X_boot = X_np[idx]

    km_boot = KMeans(n_clusters=5, random_state=None, n_init=5, max_iter=200)
    labels_boot = km_boot.fit_predict(X_boot)

    # ARI entre clustering original e bootstrap (apenas nas amostras sorteadas)
    ari = adjusted_rand_score(labels_original[idx], labels_boot)
    ari_scores.append(ari)

ari_mean = np.mean(ari_scores)
ari_std  = np.std(ari_scores)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(ari_scores, bins=25, color=PALETTE[0], edgecolor='white', alpha=0.85)
ax.axvline(ari_mean, color='red', ls='--', lw=2,
           label=f'ARI médio: {ari_mean:.3f} ± {ari_std:.3f}')
ax.axvline(0.8, color='green', ls=':', lw=1.5, alpha=0.7,
           label='Limiar "estável" (ARI ≥ 0.80)')
ax.set_title(f"Estabilidade dos clusters — Bootstrap (n={N_BOOTSTRAP})")
ax.set_xlabel("Adjusted Rand Index (ARI)")
ax.set_ylabel("Frequência")
ax.legend()

plt.tight_layout()
plt.savefig("bootstrap_stability.png", dpi=120, bbox_inches='tight')
plt.show()

print(f"ARI médio : {ari_mean:.3f}")
print(f"ARI std   : {ari_std:.3f}")
print(f"ARI mín   : {min(ari_scores):.3f} | máx: {max(ari_scores):.3f}")
if ari_mean >= 0.8:
    print("✅ Clusters estáveis (ARI ≥ 0.80) — os grupos se repetem em amostras diferentes.")
else:
    print("⚠️  Estabilidade moderada — considere testar k diferente ou mais features.")


## 8. Visualização avançada — PCA e t-SNE

> **Correção técnica importante**  
> No notebook original, PCA e t-SNE eram aplicados sobre apenas 2 variáveis (`Income_sc`, `Score_sc`). Com 2 dimensões, o PCA já é o próprio plano — não há redução real. Aplicamos aqui sobre as **4 features** usadas na modelagem completa (`Age_sc`, `Income_sc`, `Score_sc`, `Gender_Code`), onde a redução dimensional tem significado.

In [ ]:
# =============================================================
# PCA e t-SNE em 4 dimensões → 2D
# =============================================================
X_4d = df[['Age_sc', 'Income_sc', 'Score_sc', 'Gender_Code']].values

# PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_4d)

# t-SNE (perplexidade balanceada para n=200)
tsne = TSNE(n_components=2, random_state=42, perplexity=20, max_iter=1000,
            learning_rate='auto', init='pca')
X_tsne = tsne.fit_transform(X_4d)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Redução dimensional em 4 features (Age, Income, Score, Gender)")

for ax, X_2d, title, note in zip(
    axes,
    [X_pca, X_tsne],
    ["PCA — projeção linear", "t-SNE — projeção não-linear"],
    [f"Variância explicada: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}", ""],
):
    for c in sorted(df['Cluster'].unique()):
        mask = df['Cluster'] == c
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   color=PALETTE[c], label=f'C{c}: {CLUSTER_LABELS[c]}',
                   s=70, alpha=0.85, edgecolors='white', lw=0.4)
    ax.set_title(title)
    ax.set_xlabel("Componente 1")
    ax.set_ylabel("Componente 2")
    if note:
        ax.text(0.02, 0.98, note, transform=ax.transAxes,
                fontsize=9, va='top', color='gray')
    ax.legend(fontsize=7, loc='best')

plt.tight_layout()
plt.savefig("pca_tsne.png", dpi=120, bbox_inches='tight')
plt.show()

print(f"PCA — variância total explicada em 2D: {sum(pca.explained_variance_ratio_):.1%}")
print("💡 t-SNE confirma os 5 grupos — boa separação visual mesmo nas 4 dimensões.")


## 9. Insights e recomendações de negócio

Cada cluster se traduz em uma estratégia de marketing distinta.

In [ ]:
# =============================================================
# GERAÇÃO DE INSIGHTS — função robusta com base no perfil real
# =============================================================

def gerar_insights(summary_df):
    """
    Gera texto interpretativo para cada cluster a partir de suas médias.
    Mais robusto que regras fixas: baseia-se nas medianas do dataset.
    """
    income_mid = df['Annual_Income'].median()
    score_mid  = df['Spending_Score'].median()
    age_mid    = df['Age'].median()

    acoes = {
        0: ("Fidelização com custo-benefício",
            "Programa de pontos, cupons de desconto e comunicação racional sobre valor."),
        1: ("Conversão de alto potencial",
            "Campanhas premium personalizadas, experiências exclusivas, amostras grátis — eles têm renda mas não estão gastando aqui."),
        2: ("Retenção com acessibilidade",
            "Promoções de entrada, parcelamentos, combos de baixo ticket — reduzir barreira de preço."),
        3: ("Programa VIP / Upselling",
            "Clube de benefícios, pré-venda de lançamentos, personal shopper — eles são o segmento mais rentável."),
        4: ("Engajamento e ticket médio",
            "Incentivos para aumentar gasto médio: cashback, desafios de compra, mix de categorias premium."),
    }

    print("=" * 70)
    for cluster, row in summary_df.iterrows():
        acao, estrategia = acoes[cluster]
        print(f"\n🏷  Cluster {cluster} — {CLUSTER_LABELS[cluster]}")
        print(f"   Perfil: {row['N']:.0f} clientes | Idade avg {row['Idade_media']:.0f} anos "
              f"| Renda avg ${row['Renda_media']:.0f}k | Score avg {row['Score_medio']:.0f}")
        print(f"   Gênero: {row['Pct_Feminino']:.0f}% feminino")
        print(f"   ➜ Ação: {acao}")
        print(f"     {estrategia}")
    print("\n" + "=" * 70)

gerar_insights(cluster_summary)


In [ ]:
# =============================================================
# SALVAR OUTPUTS
# =============================================================
df.to_csv("cluster_assignments.csv", index=False)
cluster_summary.to_csv("cluster_summary.csv")

print("✅ Arquivos salvos:")
print("   • cluster_assignments.csv — dataset completo com coluna Cluster e Cluster_Label")
print("   • cluster_summary.csv    — médias e contagens por cluster")
print("   • eda_overview.png       — painel EDA")
print("   • clusters_scatter.png   — scatter e boxplot dos clusters")
print("   • silhouette_plot.png    — análise de coesão por amostra")
print("   • bootstrap_stability.png — validação de estabilidade")
print("   • pca_tsne.png           — visualização dimensional")
